# Practice 3: Exercise 1 — Zero-Shot Sentiment Analysis & Dataset EDA

## Overview & Purpose
This notebook demonstrates and analyzes Exercise 1 of Practice 3:
1. **Dataset Exploratory Data Analysis (EDA):** Class balance (50/50 ratio) and review length distribution on [`stanfordnlp/imdb`](https://huggingface.co/datasets/stanfordnlp/imdb).
2. **Tokenization Breakdown:** Inspecting tokenization details (tokens, token IDs, attention mask) with [`distilbert-base-uncased-finetuned-sst-2-english`](https://huggingface.co/distilbert-base-uncased-finetuned-sst-2-english).
3. **Zero-Shot Sentiment Pipeline Demo:** Running sentiment analysis on sample sentences.
4. **Baseline Evaluation Artifacts:** Visualizing baseline 5W1H metrics, ROC-AUC score (`0.9587`), and PyTorch TensorBoard log locations (produced by [`src/experiments/baseline_imdb_sentiment.py`](../src/experiments/baseline_imdb_sentiment.py)).

> **Note:** Per project rules ([`LOGGING_CHECKPOINT_RULES.md`](../agents/rules/LOGGING_CHECKPOINT_RULES.md)), notebooks do not run long evaluation loops — they load persisted JSON artifacts and plots produced by scripts.

## Roadmap Table
| Step | Description | What it does | Import path |
|:---:|:---|:---|:---|
| 1 | Dataset EDA & Visualizations | Load dataset stats, label distribution, and review length plots | [`src/data/eda_imdb.py`](../src/data/eda_imdb.py) |
| 2 | Tokenization Breakdown | Demonstrate tokenization, input IDs, and attention mask | `transformers.AutoTokenizer` |
| 3 | Zero-Shot Pipeline Demo | Run sentiment analysis pipeline on sample sentences | [`src/experiments/baseline_imdb_sentiment.py`](../src/experiments/baseline_imdb_sentiment.py) |
| 4 | Load Baseline & ROC Artifacts | Display persisted 5W1H metrics and ROC-AUC curve plot | [`experiments/results/baseline_imdb_sentiment.json`](../experiments/results/baseline_imdb_sentiment.json) |

---

## References
- **Rulebase:** [`LOGGING_CHECKPOINT_RULES.md`](../agents/rules/LOGGING_CHECKPOINT_RULES.md), [`RESULTS_REPORTING.md`](../agents/rules/RESULTS_REPORTING.md), [`NOTEBOOK_HEADER_CONVENTION.md`](../agents/rules/NOTEBOOK_HEADER_CONVENTION.md)
- **EDA Script Entry Point:** [`src/data/eda_imdb.py`](../src/data/eda_imdb.py)
- **Baseline Script Entry Point:** [`src/experiments/baseline_imdb_sentiment.py`](../src/experiments/baseline_imdb_sentiment.py)
- **Persisted Artifacts:** [`experiments/results/baseline_imdb_sentiment.json`](../experiments/results/baseline_imdb_sentiment.json), [`experiments/results/imdb_dataset_eda.json`](../experiments/results/imdb_dataset_eda.json)
- **Hugging Face Model:** [`distilbert-base-uncased-finetuned-sst-2-english`](https://huggingface.co/distilbert-base-uncased-finetuned-sst-2-english)
- **Hugging Face Dataset:** [`stanfordnlp/imdb`](https://huggingface.co/datasets/stanfordnlp/imdb)


In [ ]:
import sys
import json
import time
from pathlib import Path
import numpy as np
import torch
from IPython.display import Image, display
from transformers import AutoTokenizer, pipeline

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

device_id = 0 if torch.cuda.is_available() else -1
device_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"

print(f"Project Root: {PROJECT_ROOT}")
print(f"Device:       {device_name} (device_id={device_id})")


In [ ]:
# Step 1: Load and Display IMDB Dataset EDA Visualizations
eda_json_path = PROJECT_ROOT / "experiments" / "results" / "imdb_dataset_eda.json"

if not eda_json_path.exists():
    print("Running EDA script to generate statistics and plots...")
    from src.data.eda_imdb import run_imdb_eda
    stats = run_imdb_eda()
else:
    with open(eda_json_path, "r", encoding="utf-8") as f:
        stats = json.load(f)

print("=" * 65)
print(" IMDB DATASET EXPLORATORY DATA ANALYSIS (EDA)")
print("=" * 65)
print(f"Train Split: {stats['splits']['train']['total']:,} samples (50% Neg, 50% Pos)")
print(f"Test Split:  {stats['splits']['test']['total']:,} samples (50% Neg, 50% Pos)")
l_stats = stats['review_length_word_count_stats']
print(f"Word Count Stats (Train): Mean={l_stats['mean']:.1f}, Median={l_stats['median']:.0f}, 95th Pct={l_stats['pct_95']:.0f}, Max={l_stats['max']}")
print("=" * 65)

# Display Class Balance & Review Length Plots
print("\n--- Class Balance Distribution ---")
display(Image(filename=str(PROJECT_ROOT / "experiments" / "plots" / "imdb_label_distribution.png")))

print("\n--- Review Length Distribution (Word Count) ---")
display(Image(filename=str(PROJECT_ROOT / "experiments" / "plots" / "imdb_review_length_distribution.png")))


In [ ]:
# Step 2: Exercise 1 Tokenization Breakdown
model_name = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(model_name)

sample_text = "The movie was thrilling and visually stunning!"
tokens = tokenizer.tokenize(sample_text)
token_ids = tokenizer.convert_tokens_to_ids(tokens)
encoding = tokenizer(sample_text)

print(f"Sample Input:   '{sample_text}'")
print(f"Tokens:         {tokens}")
print(f"Token IDs:      {token_ids}")
print(f"Input IDs:      {encoding['input_ids']}")
print(f"Attention Mask: {encoding['attention_mask']}")


In [ ]:
# Step 3: Hugging Face Zero-Shot Sentiment Pipeline
clf = pipeline("sentiment-analysis", model=model_name, device=device_id)

test_sentences = [
    "The movie was thrilling and visually stunning!",
    "The plot was boring and predictable, a waste of time.",
    "It had incredible visuals, but the story felt completely hollow."
]

print("--- Zero-Shot Sentiment Predictions ---")
for text in test_sentences:
    t0 = time.time()
    res = clf(text)[0]
    dt = (time.time() - t0) * 1000.0
    print(f'Input:      "{text}"')
    print(f"Prediction: Label={res['label']}, Score={res['score']:.4f} (latency: {dt:.2f} ms)\n")


---

# DF40 DATA EDA

## Overview & Purpose
This section contains comprehensive Exploratory Data Analysis (EDA) for the DF40 deepfake detection dataset:
1. **Dataset Source & Structure**: Analysis of Hugging Face sources and local organization
2. **Dataset Statistics**: Sample counts, video counts, and distribution across manipulation methods
3. **Label Schema**: Determination of REAL vs FAKE labels from test manifest
4. **Class Distribution**: Analysis of severe class imbalance (95% FAKE vs 5% REAL)
5. **Manipulation Distribution**: Distribution across 41 manipulation methods
6. **Video-Level Statistics**: Frame counts and video organization
7. **Frame/Image Statistics**: Resolution, aspect ratio, file properties
8. **Data Quality Analysis**: Corrupted files, invalid samples, trainability
9. **Duplicate Analysis**: Exact duplicate detection and cross-split leakage
10. **Visual Sample Inspection**: Sample file paths for visual inspection
11. **EDA Findings**: Critical, high, medium, and low priority findings

> **Note:** All statistics are calculated from the actual downloaded DF40 dataset metadata located in `src/data/DF40/metadata/`.

## DF40 Dataset Sources
- **Training Dataset**: `https://huggingface.co/datasets/ManhQuangAI/DF40_train`
- **Test Dataset**: `https://huggingface.co/datasets/ManhQuangAI/df40-test-data-v3`
- **Model Reference**: `https://huggingface.co/ManhQuangAI/dinov3-deepfake-detection`

In [ ]:
# DF40 EDA - Dataset Statistics and Label Schema
import pandas as pd
import json
from pathlib import Path

# Define PROJECT_ROOT for DF40 cells (works in notebook context)
PROJECT_ROOT = Path("..").resolve()

# DF40 paths
DF40_ROOT = PROJECT_ROOT / "src" / "data" / "DF40"
TRAIN_ROOT = DF40_ROOT / "train"
TEST_ROOT = DF40_ROOT / "test"
METADATA_ROOT = DF40_ROOT / "metadata"
EDA_ROOT = DF40_ROOT / "eda"

print("=" * 65)
print(" DF40 DATASET STATISTICS")
print("=" * 65)

# Load training summary
summary_file = METADATA_ROOT / 'training_summary.json'
with open(summary_file, 'r') as f:
    summary = json.load(f)

print(f"Dataset Location: {DF40_ROOT}")
print(f"Total Samples: {summary['total_samples']:,}")
print(f"Total Videos: {summary['total_videos']:,}")
print(f"Real Samples: {summary['real_samples']:,} ({summary['real_samples']/summary['total_samples']*100:.2f}%)")
print(f"Fake Samples: {summary['fake_samples']:,} ({summary['fake_samples']/summary['total_samples']*100:.2f}%)")
print(f"Manipulation Methods: {summary['manipulation_methods']}")
print(f"Resolution: {summary['resolution']}")
print(f"Color Mode: {summary['color_mode']}")
print(f"Format: {summary['format']}")
print("=" * 65)

In [ ]:
# DF40 EDA - Class and Manipulation Distribution
import pandas as pd
from pathlib import Path

# Define paths for this cell
PROJECT_ROOT = Path("..").resolve()
DF40_ROOT = PROJECT_ROOT / "src" / "data" / "DF40"
METADATA_ROOT = DF40_ROOT / "metadata"

print("=" * 65)
print(" CLASS AND MANIPULATION DISTRIBUTION")
print("=" * 65)

# Load class distribution
class_df = pd.read_csv(METADATA_ROOT / 'class_distribution.csv')
print("\nClass Distribution:")
print(class_df.to_string(index=False))

# Load manipulation distribution
manip_df = pd.read_csv(METADATA_ROOT / 'manipulation_distribution.csv')
print(f"\nTotal Manipulation Methods: {len(manip_df)}")
print("\nTop 15 Manipulation Methods:")
print(manip_df.head(15).to_string(index=False))

print("=" * 65)

In [ ]:
# DF40 EDA - Video-Level and Data Quality Analysis
import pandas as pd
from pathlib import Path

# Define paths for this cell
PROJECT_ROOT = Path("..").resolve()
DF40_ROOT = PROJECT_ROOT / "src" / "data" / "DF40"
METADATA_ROOT = DF40_ROOT / "metadata"

print("=" * 65)
print(" VIDEO-LEVEL AND DATA QUALITY ANALYSIS")
print("=" * 65)

# Load video metadata
video_df = pd.read_csv(METADATA_ROOT / 'df40_video_metadata.csv')
print(f"\nTotal Videos: {len(video_df)}")
print(f"\nFrames per Video Statistics:")
print(video_df['frame_count'].describe())

print(f"\nVideo Distribution by Split:")
print(video_df['split_source'].value_counts())

# Load quality report
quality_df = pd.read_csv(METADATA_ROOT / 'quality_report.csv')
print(f"\nQuality Report:")
print(quality_df.to_string(index=False))

print("=" * 65)

In [ ]:
# DF40 EDA - Duplicate and Trainable Data Analysis
import pandas as pd
from pathlib import Path

# Define paths for this cell
PROJECT_ROOT = Path("..").resolve()
DF40_ROOT = PROJECT_ROOT / "src" / "data" / "DF40"
METADATA_ROOT = DF40_ROOT / "metadata"

print("=" * 65)
print(" DUPLICATE AND TRAINABLE DATA ANALYSIS")
print("=" * 65)

# Load duplicate information
duplicates_df = pd.read_csv(METADATA_ROOT / 'duplicates.csv')
print(f"Total Duplicate Samples: {len(duplicates_df)}")
if len(duplicates_df) > 0:
    print("\nDuplicate Distribution by Original Split:")
    print(duplicates_df['split_source'].value_counts())
    print("\nDuplicate Distribution by Manipulation:")
    print(duplicates_df['manipulation'].value_counts().head(10))
else:
    print("No exact duplicates found.")

# Load cross-split duplicates
cross_split_df = pd.read_csv(METADATA_ROOT / 'cross_split_duplicates.csv')
print(f"\nCross-Split Duplicate Samples: {len(cross_split_df)}")
print(f"Cross-Split Duplicate Percentage: {len(cross_split_df) / 258375 * 100:.2f}%")

# Load final training metadata for trainable analysis
final_df = pd.read_csv(METADATA_ROOT / 'final_training_metadata.csv')
trainable_samples = final_df[final_df['is_trainable'] == True]
print(f"\nTotal Trainable Samples: {len(trainable_samples)}")
print(f"Trainable Percentage: {len(trainable_samples) / len(final_df) * 100:.2f}%")

print(f"\nTrainable by Split:")
print(trainable_samples['split'].value_counts())

print("=" * 65)

In [ ]:
# DF40 EDA - EDA Findings
print("=" * 65)
print(" EDA FINDINGS")
print("=" * 65)

print("CRITICAL:")
print("  - Severe class imbalance: 95% FAKE vs 5% REAL")
print("  - Duplicate files cross splits: 45,324 duplicates detected (17.54%)")
print("  - Multiple manipulation methods with significant imbalance")

print("\nHIGH:")
print("  - Class imbalance may bias model toward FAKE predictions")
print("  - Minor manipulation methods (<1% each) may not be learned effectively")
print("  - Cross-split duplicates indicate frame-level correlation within videos")

print("\nMEDIUM:")
print("  - Dataset organization requires metadata-driven loading")
print("  - Video-level splitting successfully prevents video leakage")
print("  - Image properties are consistent (256x256, RGB, PNG)")

print("\nLOW:")
print("  - No corrupted files detected")
print("  - All samples are trainable (100%)")
print("  - Clean data quality (EXCELLENT)")

print("\nVIDEO LEAKAGE:")
print("  - Train ∩ Validation: 0 videos ✅")
print("  - Train ∩ Test: 0 videos ✅")
print("  - Validation ∩ Test: 0 videos ✅")
print("  - Video leakage check: PASS")

print("=" * 65)

In [ ]:
# DF40 EDA - Display EDA Visualizations
from IPython.display import Image, display
from pathlib import Path

# Define paths for this cell
PROJECT_ROOT = Path("..").resolve()
DF40_ROOT = PROJECT_ROOT / "src" / "data" / "DF40"
EDA_ROOT = DF40_ROOT / "eda"

print("=" * 65)
print(" EDA VISUALIZATIONS")
print("=" * 65)

# Display EDA figures
figures = [
    ('class_distribution.png', 'Class Distribution'),
    ('split_distribution.png', 'Split Distribution'),
    ('manipulation_distribution.png', 'Manipulation Distribution'),
    ('resolution_distribution.png', 'Resolution Distribution'),
    ('aspect_ratio_distribution.png', 'Aspect Ratio Distribution'),
    ('file_size_distribution.png', 'File Size Distribution'),
    ('class_by_split_distribution.png', 'Class by Split Distribution'),
    ('fake_manipulation_distribution.png', 'Fake Manipulation Distribution')
]

for figure_file, title in figures:
    figure_path = EDA_ROOT / 'figures' / figure_file
    if figure_path.exists():
        print(f"\n--- {title} ---")
        display(Image(filename=str(figure_path)))
    else:
        print(f"Figure not found: {figure_file}")

print("=" * 65)

In [ ]:
# Step 4: Load Persisted Baseline Results & ROC Curve Visualization
artifact_path = PROJECT_ROOT / "experiments" / "results" / "baseline_imdb_sentiment.json"

if not artifact_path.exists():
    print(f"Artifact not found at {artifact_path}. Please run:")
    print("  python -m src.experiments.baseline_imdb_sentiment --eval-imdb")
else:
    with open(artifact_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    print("=" * 65)
    print(" PERSISTED 5W1H METADATA & BASELINE EVALUATION RESULTS")
    print("=" * 65)
    meta = data["metadata_5w1h"]
    print(f"Who:   {meta['who']}")
    print(f"What:  {meta['what']}")
    print(f"When:  {meta['when']}")
    print(f"Where: {meta['where']}")
    print(f"Why:   {meta['why']}")
    print(f"How:   {meta['how']}")

    print("\n--- Metric Summary ---")
    eval_m = data["evaluation"]
    print(f"Evaluated Test Samples:  {eval_m['num_test_samples']:,}")
    print(f"Majority-Class Floor:     {eval_m['majority_class_accuracy'] * 100.0:.2f}%")
    print(f"Zero-Shot Model Accuracy: {eval_m['zero_shot_accuracy'] * 100.0:.2f}%")
    if "zero_shot_roc_auc" in eval_m and not np.isnan(eval_m["zero_shot_roc_auc"]):
        print(f"Zero-Shot ROC-AUC Score:  {eval_m['zero_shot_roc_auc']:.4f}")
    print(f"Total Evaluation Time:   {eval_m['eval_time_seconds']:.2f} s")
    print(f"TensorBoard Log Dir:     {eval_m.get('tensorboard_log_dir', 'N/A')}")
    print("=" * 65)

roc_plot_path = PROJECT_ROOT / "experiments" / "plots" / "baseline_zero_shot_roc_curve.png"
if roc_plot_path.exists():
    print("\n--- Zero-Shot ROC Curve Plot ---")
    display(Image(filename=str(roc_plot_path)))